In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re as re

In [55]:
cases=pd.read_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/raw/cases.csv')

In [56]:
print(f" {len(cases)}")

 96966


In [58]:
cases['identifier'] = (cases['case_number'] + '_' + 
                       cases['district_id'] + '_' + 
                       cases['case_name'].str[:3].str.lower().str.strip())


In [59]:
print(cases['identifier'].isna().sum())

96


In [60]:
# Build a mapping of case_number -> first non-null case_name
case_name_map = cases.dropna(subset=['case_name']).drop_duplicates('case_number').set_index('case_number')['case_name']

# Fill nulls using the mapping
cases.loc[cases['case_name'].isna(), 'case_name'] = cases.loc[cases['case_name'].isna(), 'case_number'].map(case_name_map)

print("Remaining nulls:", cases['case_name'].isna().sum())

Remaining nulls: 78


In [61]:
cases['identifier'] = (cases['case_number'] + '_' + 
                       cases['district_id'] + '_' + 
                       cases['case_name'].str[:3].str.lower().str.strip())

In [62]:
print(cases['identifier'].isna().sum())

78


In [63]:
mask = cases['identifier'].isna()
cases.loc[mask, 'identifier'] = (
    cases.loc[mask, 'case_number'] + '_' + 
    cases.loc[mask, 'district_id'] + '_noname'
)

In [64]:
print(cases['identifier'].isna().sum())

0


In [65]:
dupes = cases[cases['identifier'].duplicated(keep=False)]
print(f"Total duplicate rows: {len(dupes)}")
dupes.sort_values('identifier').to_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/raw/duplicate_identifiers_v3.csv', index=False)

Total duplicate rows: 163


In [67]:
# Count non-null values per row as a proxy for "most complete"
cases['completeness'] = cases.notna().sum(axis=1)

# Sort by completeness descending so the most complete row comes first
# Then drop duplicates keeping the first (most complete) row
cases = (cases
    .sort_values('completeness', ascending=False)
    .drop_duplicates(subset='identifier', keep='first')
    .drop(columns='completeness')
)

print(f"Cases after deduplication: {len(cases)}")

Cases after deduplication: 96881


In [68]:
dupes = cases[cases['identifier'].duplicated(keep=False)]
print(f"Total duplicate rows: {len(dupes)}")
dupes.sort_values('identifier').to_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/raw/duplicate_identifiers_v3.csv', index=False)

Total duplicate rows: 0


In [54]:
cases.to_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/cases/cases_clean.csv', index=False)